my 1st attempt at implimenting 
#### 'http://arxiv.org/abs/1407.4419v2' 

In [ ]:
pip install qiskit

In [ ]:
import qiskit
qiskit.__version__

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, partial_trace, entropy
import matplotlib.pyplot as plt
import random

In [ ]:
def apply_random_gate(qc, gate_set):
    gate = random.choice(gate_set)
    q1 = random.randint(0, qc.num_qubits - 1)
    q2 = random.randint(0, qc.num_qubits - 1)
    while q2 == q1:
        q2 = random.randint(0, qc.num_qubits - 1)
    qc.cx(q1, q2)
    if gate =='1' :
        qc.x(q1)
        qc.h(q1)

    elif gate == '2':
        qc.h(q1)
        qc.t(q1)

def half_chain_entropy(statevec, L):
    rho_A = partial_trace(statevec, range(L//2, L))
    return entropy(rho_A)

def cooling_algorithm(state, L, gate_set, beta, max_steps=1000):
    current_state = state
    current_entropy = half_chain_entropy(current_state, L)
    entropy_evolution = [current_entropy]
    accepted = 0

    

    for step in range(max_steps):
        
        qc_new = QuantumCircuit(L)
        
        gate = random.choice(gate_set)
        q1 = random.randint(0, L - 1)
        apply_random_gate(qc_new,gate)
 

        new_state = current_state.evolve(qc_new)
        new_entropy = half_chain_entropy(new_state, L)

        delta_S = new_entropy - current_entropy

        if delta_S < 0 or np.random.rand() < np.exp(-beta * delta_S):
            current_state = new_state
            current_entropy = new_entropy
            accepted += 1
        
        entropy_evolution.append(current_entropy)
        
        if np.isclose(current_entropy, 0, atol=1e-2):
            print('required tolerance achieved')
            break
    
    return entropy_evolution, step + 1, accepted,current_state


def entanglement_heating(L, gate_set, steps=None):
    if steps is None:
        steps = L**2  

    qc = QuantumCircuit(L)
    for _ in range(steps):
        apply_random_gate(qc, gate_set)

    
    return Statevector.from_instruction(qc)


In [ ]:
L = 6             
L2 = L**2         
cooling_steps = 1000
beta = 500.0       
gate_set1 = ['1']
gate_set2 = ['2']
seed = 45
np.random.seed(seed)
random.seed(seed)

state = entanglement_heating(L, gate_set2)
print("Entropy after heating:", half_chain_entropy(state, L))
entropy_trace1, steps1, acc1,f_state = cooling_algorithm(state, L, gate_set1, beta, cooling_steps)
print("Entropy after cooling:", half_chain_entropy(f_state, L))
if entropy_trace[-1]- entropy_trace[-3] <=0.5:
    entropy_trace2, steps2, acc2,f_state2 = cooling_algorithm(f_state, L, gate_set2, beta, cooling_steps)
    print("Entropy after 2nd cooling:", half_chain_entropy(f_state2, L))


entropy_trace=entropy_trace1+entropy_trace2
steps =steps1+steps2
plt.figure(figsize=(8,5))
plt.plot(entropy_trace, label=f"β={beta}, steps={steps}")
plt.xlabel("Cooling Iteration")
plt.ylabel("Half-chain Entanglement Entropy")
plt.title(f"Entanglement Cooling (L={L}) with Gate Set: {gate_set1}")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()